In [9]:
import pandas as pd

df = pd.read_csv("downloaded_custom_dataset.csv")

df.drop(['DATAFLOW', 'LAST UPDATE', 'freq', 'unit','CONF_STATUS'], axis=1, inplace=True) 
df.rename(columns={'TIME_PERIOD': 'year', 'geo':'country', 'hhtyp':'hhtype'}, inplace=True)


# ============================================================
# Preparation of dataset 1
# ============================================================
# Filter hhtype = Total
df_total = df[df["hhtype"].str.lower() == "total"].copy()

# Remove missing values 
df_total = df_total.dropna(subset=["OBS_VALUE"])

# Remove aggregate observations
countries_to_remove = [
    "Euro area (EA11-1999, EA12-2001, EA13-2007, EA15-2008, EA16-2009, EA17-2011, EA18-2014, EA19-2015, EA20-2023, EA21-2026)",
    "European Union - 15 countries (1995-2004)",
    "European Union - 25 countries (2004-2006)",
    "European Union - 27 countries (2007-2013)",
    "European Union - 27 countries (from 2020)",
    "European Union - 28 countries (2013-2020)"
]

df_total = df_total[~df_total["country"].isin(countries_to_remove)]

# Create the final dataframe structure
df_points = df_total.copy()
df_points = df_points[["year", "country", "OBS_VALUE"]]
df_points["series"] = ""
df_points = df_points.rename(columns={"OBS_VALUE": "value"})
# Add the yearly average values to the dataset
df_mean = (
    df_total
    .groupby("year", as_index=False)["OBS_VALUE"]
    .mean()
)
df_mean["series"] = "average"
df_mean["country"] = "Average"
df_mean = df_mean.rename(columns={"OBS_VALUE": "value"})

df_viz_1 = pd.concat(
    [df_points, df_mean],
    ignore_index=True
)
df_viz_1 = df_viz_1.sort_values(["year", "series"])
df_viz_1.to_csv('dataset_for_first_viz.csv', index=False)



# ============================================================
# Preparation of dataset 2
# ============================================================
# Filter hhtype 
hh_with = "All types with dependent children"
hh_without = "All types without dependent children"
df_hh = df[df["hhtype"].isin([hh_with, hh_without])].copy()

# Remove aggregate observations and missing values
df_hh = df_hh[~df_hh["country"].isin(countries_to_remove)]
df_hh = df_hh.dropna(subset=["OBS_VALUE"])

# Compute averages by year and household type
mean_by_year_type = (
    df_hh
    .groupby(["year", "hhtype"], as_index=False)["OBS_VALUE"]
    .mean()
)

# Each hhtype should represent a column
df_viz_2 = mean_by_year_type.pivot(index="year", columns="hhtype", values="OBS_VALUE")

df_viz_2 = df_viz_2.rename(columns={"All types with dependent children":"Households with dependent children",
                      "All types without dependent children":"Households without dependent children"})

df_viz_2.to_csv('dataset_for_second_viz.csv', index=True)


